# NB_00_RP_37_SOURCE_EXTRACTION

This notebook converts one engineering source into structured, reviewable Reading Point inputs.

**Source workflow**

```text
Engineering source
        ↓
Measured engineering states
        ↓
Target specifications
        ↓
Engineering constraints
        ↓
Engineering refinements
        ↓
Candidate Reading Point dialogue
```

This v1 is grounded in Dan Becker's presentation:

> *Achieving 1% Assay of Special Nuclear Materials in 2 Minutes with Microcalorimeter-Array Gamma-Ray Spectroscopy*  
> ARPA-E Fission Annual Meeting, October 1–2, 2025.

The notebook does not infer missing measurements. Every extracted item should remain traceable to a source page.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import json
import zipfile

try:
    import yaml
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pyyaml"],
        check=True,
    )
    import yaml

try:
    import pandas as pd
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pandas"],
        check=True,
    )
    import pandas as pd

from IPython.display import Markdown, display

NOTEBOOK_ID = "NB_00_RP_37_SOURCE_EXTRACTION"
NOTEBOOK_VERSION = "1.2.0"
REPOSITORY = "sensors-becker"

OUTPUT_DIRECTORY = Path("outputs/source_extraction/becker_2025_arpa_e")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": REPOSITORY,
    "output_directory": str(OUTPUT_DIRECTORY),
}


## Source Identity

The source record establishes provenance before any engineering statement is written.


In [ ]:
SOURCE = {
    "source_id": "BECKER_2025_ARPA_E_MICROCALORIMETER_ASSAY",
    "title": (
        "Achieving 1% Assay of Special Nuclear Materials in 2 Minutes "
        "with Microcalorimeter-Array Gamma-Ray Spectroscopy"
    ),
    "author": "Dan Becker",
    "organization": "University of Colorado",
    "event": "ARPA-E Fission Annual Meeting",
    "date": "2025-10-01/2025-10-02",
    "source_file": "Daniel Becker (1).pdf",
    "page_count": 16,
    "engineering_object": "Microcalorimeter",
    "engineering_direction": "Toward next-generation microcalorimeters.",
}

SOURCE


## Extraction Schema

Each item records:

- the source page;
- the source-derived engineering statement;
- the engineering category;
- any reported value, unit, and comparison state;
- a short note explaining how the item is used.

The categories in this notebook are:

```text
measured_state
target_specification
engineering_constraint
engineering_refinement
deployment_requirement
```


In [ ]:
@dataclass(frozen=True)
class SourceExtraction:
    extraction_id: str
    category: str
    statement: str
    page: int
    metric: str | None = None
    value: float | int | str | None = None
    unit: str | None = None
    comparison_state: str | None = None
    note: str = ""

    def validate(self) -> None:
        allowed = {
            "measured_state",
            "target_specification",
            "engineering_constraint",
            "engineering_refinement",
            "deployment_requirement",
        }
        if self.category not in allowed:
            raise ValueError(f"Unsupported category: {self.category}")
        if self.page < 1 or self.page > SOURCE["page_count"]:
            raise ValueError(f"Invalid source page: {self.page}")
        if not self.statement.strip():
            raise ValueError("statement is required")


## Source-Derived Extractions

The following entries are derived directly from the presentation. Edit or extend this cell as additional sources are reviewed.


In [ ]:
EXTRACTIONS = [
    SourceExtraction(
        extraction_id="MS_001",
        category="measured_state",
        statement="Current CURIE detector speed is compatible with 140 photons per second.",
        page=12,
        metric="per_detector_count_rate",
        value=140,
        unit="counts_per_second",
        comparison_state="current",
        note="Reported as within 1.4× of the 200 cps target.",
    ),
    SourceExtraction(
        extraction_id="MS_002",
        category="measured_state",
        statement="Current pulse fall time to 1% of peak is approximately 1.5 milliseconds.",
        page=16,
        metric="fall_time_to_1_percent",
        value=1.5,
        unit="milliseconds",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="MS_003",
        category="measured_state",
        statement="Current athermal tails are negligible.",
        page=16,
        metric="athermal_tails",
        value="negligible",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="MS_004",
        category="measured_state",
        statement="Current energy resolution is approximately 150 electronvolts at 100 kiloelectronvolts.",
        page=16,
        metric="energy_resolution_at_100_keV",
        value=150,
        unit="electronvolts",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="MS_005",
        category="measured_state",
        statement="The all-silicon detector architecture has produced approximately a 14× speed improvement.",
        page=16,
        metric="speed_improvement",
        value=14,
        unit="times",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="TS_001",
        category="target_specification",
        statement="Increase per-detector count rate to 200 counts per second.",
        page=7,
        metric="per_detector_count_rate",
        value=200,
        unit="counts_per_second",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="TS_002",
        category="target_specification",
        statement="Maintain energy resolution below 100 electronvolts at 100 kiloelectronvolts.",
        page=7,
        metric="energy_resolution_at_100_keV",
        value="<100",
        unit="electronvolts",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="TS_003",
        category="target_specification",
        statement="Reduce pulse fall time to 750 microseconds.",
        page=7,
        metric="fall_time_to_1_percent",
        value=750,
        unit="microseconds",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="TS_004",
        category="target_specification",
        statement="Enable assay of complex special nuclear material mixtures to within 1% accuracy in 2 minutes.",
        page=2,
        metric="assay_accuracy_and_time",
        value="1% in 2 minutes",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="EC_001",
        category="engineering_constraint",
        statement="Detector decay time limits count rate.",
        page=3,
        metric="decay_time",
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="EC_002",
        category="engineering_constraint",
        statement="The pre-CURIE membrane architecture is finicky to assemble, fragile, and limited in its ability to increase thermal conductance G.",
        page=9,
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="EC_003",
        category="engineering_constraint",
        statement="Energy resolution degraded as detector speed increased.",
        page=12,
        metric="energy_resolution_at_100_keV",
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="ER_001",
        category="engineering_refinement",
        statement="Use an all-silicon, membrane-free detector architecture.",
        page=10,
        comparison_state="refinement",
        note="Presented as forgiving to assemble, physically robust, and providing access to a wide range of G.",
    ),
    SourceExtraction(
        extraction_id="ER_002",
        category="engineering_refinement",
        statement="Lower the operating temperature and superconducting critical temperature Tc.",
        page=14,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="ER_003",
        category="engineering_refinement",
        statement="Increase coupling to silicon.",
        page=14,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="ER_004",
        category="engineering_refinement",
        statement="Solve the absorber manufacturing problem.",
        page=15,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="DR_001",
        category="deployment_requirement",
        statement="Validate detector performance with relevant samples from INL's Hot Fuel Examination Facility.",
        page=7,
        comparison_state="deployment",
    ),
    SourceExtraction(
        extraction_id="DR_002",
        category="deployment_requirement",
        statement="Define requirements for product assembly.",
        page=15,
        comparison_state="deployment",
    ),
]

for item in EXTRACTIONS:
    item.validate()

len(EXTRACTIONS)


## Review Extracted Engineering Content

The table is the primary human-review checkpoint. It should be inspected before any Reading Point YAML is generated.


In [ ]:
extraction_table = pd.DataFrame(asdict(item) for item in EXTRACTIONS)
extraction_table[
    [
        "extraction_id",
        "category",
        "page",
        "metric",
        "value",
        "unit",
        "statement",
    ]
]


## Current-to-Target Comparison

Only metrics with explicit current and target states are compared here.


In [ ]:
comparison_rows = [
    {
        "metric": "Per-detector count rate",
        "current": "140 cps",
        "target": "200 cps",
        "source_pages": "12, 16 / 7",
    },
    {
        "metric": "Energy resolution at 100 keV",
        "current": "150 eV",
        "target": "<100 eV",
        "source_pages": "16 / 7",
    },
    {
        "metric": "Fall time to 1% of peak",
        "current": "~1.5 ms",
        "target": "750 µs",
        "source_pages": "16 / 7",
    },
    {
        "metric": "Athermal tails",
        "current": "Negligible",
        "target": "Negligible",
        "source_pages": "16 / 7",
    },
]

comparison_table = pd.DataFrame(comparison_rows)
comparison_table


## Candidate Source-Derived Reading Point

The Reading Point grammar remains in metadata. The visible dialogue carries source-specific engineering content.

```text
Current detector performance
        ↓
Target detector performance
        ↓
Priority detector refinements
        ↓
Engineering sessions
```


In [ ]:
READING_POINT_CANDIDATE = {
    "reading_point_id": "RP_37",
    "engineering_object": "Microcalorimeter",
    "source_id": SOURCE["source_id"],
    "grammar": {
        "A": "Measured Engineering Improvement informs Leading Specifications.",
        "B": "Leading Specifications direct Engineering Priorities.",
        "C": "Engineering Priorities prepare Engineering Sessions.",
    },
    "dialogue": [
        {
            "order": "A",
            "concept": "Leading Specifications",
            "title": "Detector Targets: Microcalorimeters",
            "first_label": "140 cps · 150 eV · ~1.5 ms",
            "second_label": "200 cps · <100 eV · 750 µs",
            "supporting_context": [
                "Negligible Athermal Tailing",
                "1% Assay in 2 min",
            ],
            "engineering_statement": (
                "Measured detector performance specifies the next detector-performance targets."
            ),
        },
        {
            "order": "B",
            "concept": "Engineering Priorities",
            "title": "Detector Refinements: Microcalorimeters",
            "first_label": "200 cps · <100 eV · 750 µs",
            "second_label": "Lower Tc · Increase Si Coupling",
            "supporting_context": [
                "All-Silicon Architecture",
                "Absorber Manufacturing",
            ],
            "engineering_statement": (
                "Target detector performance directs detector-refinement priorities."
            ),
        },
        {
            "order": "C",
            "concept": "Engineering Sessions",
            "title": "Detector Engineering Sessions: Microcalorimeters",
            "first_label": "Lower Tc · Increase Si Coupling",
            "second_label": "Fabricate · Characterize · Validate",
            "supporting_context": [
                "INL Sample Evaluation",
                "Assembly Requirements",
            ],
            "engineering_statement": (
                "Detector-refinement priorities prepare fabrication, characterization, "
                "and validation sessions."
            ),
        },
    ],
}

READING_POINT_CANDIDATE


## Export Source Record and RP_37 Specifications

The notebook exports the reviewed source record and directly generates the three cumulative RP_37 YAML files used by `NB_TEMPLATE.ipynb`.

- `RP_37_A.yaml` contains dialogue A.
- `RP_37_B.yaml` contains dialogues A and B.
- `RP_37_C.yaml` contains dialogues A, B, and C.

No intermediate `NB_01` notebook is required.


In [ ]:
source_record = {
    "source": SOURCE,
    "extractions": [asdict(item) for item in EXTRACTIONS],
    "current_to_target": comparison_rows,
    "reading_point_candidate": READING_POINT_CANDIDATE,
}

json_path = OUTPUT_DIRECTORY / "becker_2025_source_extraction.json"
yaml_path = OUTPUT_DIRECTORY / "becker_2025_source_extraction.yaml"
review_path = OUTPUT_DIRECTORY / "becker_2025_source_extraction.md"
candidate_path = OUTPUT_DIRECTORY / "RP_37_SOURCE_DERIVED.yaml"

json_path.write_text(
    json.dumps(source_record, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
yaml_path.write_text(
    yaml.safe_dump(
        source_record,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    ),
    encoding="utf-8",
)
candidate_path.write_text(
    yaml.safe_dump(
        READING_POINT_CANDIDATE,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    ),
    encoding="utf-8",
)

review_lines = [
    f"# Source Extraction: {SOURCE['author']}",
    "",
    f"**Source:** {SOURCE['title']}",
    "",
    "## Current-to-Target Comparison",
    "",
    comparison_table.to_markdown(index=False),
    "",
    "## Engineering Constraints",
    "",
]
for item in EXTRACTIONS:
    if item.category == "engineering_constraint":
        review_lines.append(f"- Page {item.page}: {item.statement}")

review_lines.extend(["", "## Engineering Refinements", ""])
for item in EXTRACTIONS:
    if item.category == "engineering_refinement":
        review_lines.append(f"- Page {item.page}: {item.statement}")

review_lines.extend(
    [
        "",
        "## Source-Derived RP_37 Sequence",
        "",
        "```text",
        "140 cps · 150 eV · ~1.5 ms",
        "        ↓",
        "200 cps · <100 eV · 750 µs",
        "        ↓",
        "Lower Tc · Increase Si Coupling",
        "        ↓",
        "Fabricate · Characterize · Validate",
        "```",
        "",
        "*Admissible generalizations trail leading specifications.*",
    ]
)

review_path.write_text(
    "\n".join(review_lines) + "\n",
    encoding="utf-8",
)

REPOSITORY_GRAMMAR = [
    "Engineering Object specifies Engineering System.",
    "Engineering System produces Measured Engineering States.",
    "Measurement records Measured Engineering States.",
    "Measured Engineering States identify Engineering Constraints.",
    "Engineering Constraints direct Engineering Refinements.",
    "Engineering Refinements support Measured Engineering Improvement.",
    "Measured Engineering Improvement informs Leading Specifications.",
    "Leading Specifications direct Engineering Priorities.",
    "Engineering Priorities prepare Engineering Sessions.",
    "Engineering Sessions produce Engineering Records.",
    "Engineering Records support Engineering Reports.",
    "Engineering Reports support Repository Contributions.",
    "Repository Contributions support Repository Development.",
    "Repository Development supports Continued Specification.",
    "Continued Specification supports Engineering Object.",
]

GRAMMAR_BY_STAGE = {
    "A": "Measured Engineering Improvement informs Leading Specifications.",
    "B": "Leading Specifications direct Engineering Priorities.",
    "C": "Engineering Priorities prepare Engineering Sessions.",
}

STAGE_CONFIGURATION = {
    "A": {
        "notebook_id": "NB_37_A_DETECTOR_TARGETS",
        "inherited_from": "NB_29_C_MEASURED_ENGINEERING_IMPROVEMENT",
        "natural_foundation": "Measured Engineering Improvement",
        "engineering_objective": (
            "Specify source-derived detector-performance targets."
        ),
        "forward_context": "Engineering Priorities",
        "completion_status": "developing",
    },
    "B": {
        "notebook_id": "NB_37_B_DETECTOR_REFINEMENTS",
        "inherited_from": "NB_37_A_DETECTOR_TARGETS",
        "natural_foundation": "Leading Specifications",
        "engineering_objective": (
            "Direct detector-refinement priorities from source-derived targets."
        ),
        "forward_context": "Engineering Sessions",
        "completion_status": "developing",
    },
    "C": {
        "notebook_id": "NB_37_C_ENGINEERING_SESSIONS",
        "inherited_from": "NB_37_B_DETECTOR_REFINEMENTS",
        "natural_foundation": "Engineering Priorities",
        "engineering_objective": (
            "Prepare fabrication, characterization, and validation sessions."
        ),
        "forward_context": "Engineering Records",
        "completion_status": "complete",
    },
}


def cumulative_rp_specification(stage: str) -> dict[str, Any]:
    stage_order = {"A": 1, "B": 2, "C": 3}
    if stage not in stage_order:
        raise ValueError(f"Unsupported RP_37 stage: {stage}")

    included_source = READING_POINT_CANDIDATE["dialogue"][: stage_order[stage]]
    included_dialogue = []

    for index, source_item in enumerate(included_source):
        dialogue_stage = source_item["order"]
        dialogue_status = (
            "candidate"
            if index == len(included_source) - 1
            else "admitted"
        )

        included_dialogue.append(
            {
                "order": dialogue_stage,
                "artifact_id": (
                    f"37_{dialogue_stage}_"
                    f"{source_item['concept'].lower().replace(' ', '_')}_trail"
                ),
                "concept": source_item["concept"],
                "title": source_item["title"],
                "first_label": source_item["first_label"],
                "second_label": source_item["second_label"],
                "supporting_context": source_item["supporting_context"],
                "engineering_statement": source_item["engineering_statement"],
                "status": dialogue_status,
            }
        )

    configuration = STAGE_CONFIGURATION[stage]

    return {
        "identity": {
            "notebook_id": configuration["notebook_id"],
            "reading_point": "RP_37",
            "stage": stage,
            "version": "1.0.0",
            "status": "candidate",
        },
        "reading_point": {
            "inherited_from": configuration["inherited_from"],
            "natural_foundation": configuration["natural_foundation"],
            "engineering_objective": configuration["engineering_objective"],
            "engineering_statements": [
                item["engineering_statement"]
                for item in included_dialogue
            ],
            "repository_grammar": REPOSITORY_GRAMMAR,
            "forward_context": configuration["forward_context"],
            "status": configuration["completion_status"],
        },
        "dialogue": included_dialogue,
        "engineering_object": SOURCE["engineering_object"],
        "engineering_direction": SOURCE["engineering_direction"],
        "source": {
            "source_id": SOURCE["source_id"],
            "title": SOURCE["title"],
            "author": SOURCE["author"],
            "organization": SOURCE["organization"],
            "event": SOURCE["event"],
            "date": SOURCE["date"],
            "source_file": SOURCE["source_file"],
        },
        "source_engineering_states": {
            "measured_states": [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == "measured_state"
            ],
            "target_specifications": [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == "target_specification"
            ],
            "engineering_constraints": [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == "engineering_constraint"
            ],
            "engineering_refinements": [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == "engineering_refinement"
            ],
            "deployment_requirements": [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == "deployment_requirement"
            ],
        },
        "footer": (
            "Admissible generalizations trail leading specifications."
        ),
    }


rp_paths = {}
rp_specifications = {}

for stage in ("A", "B", "C"):
    specification = cumulative_rp_specification(stage)
    path = OUTPUT_DIRECTORY / f"RP_37_{stage}.yaml"

    path.write_text(
        yaml.safe_dump(
            specification,
            sort_keys=False,
            allow_unicode=True,
            width=100,
        ),
        encoding="utf-8",
    )

    rp_paths[stage] = path
    rp_specifications[stage] = specification

readme_path = OUTPUT_DIRECTORY / "RP_37_README.md"
readme_path.write_text(
    "# RP_37 — Source-Derived Microcalorimeter Reading Point\n\n"
    "A: current detector performance → target detector performance\n\n"
    "B: target detector performance → detector-refinement priorities\n\n"
    "C: detector-refinement priorities → engineering sessions\n\n"
    "Generated directly by NB_00_RP_37_SOURCE_EXTRACTION.\n",
    encoding="utf-8",
)

zip_path = OUTPUT_DIRECTORY / "NB_00_RP_37_SOURCE_EXTRACTION.zip"
bundle_paths = [
    json_path,
    yaml_path,
    review_path,
    candidate_path,
    rp_paths["A"],
    rp_paths["B"],
    rp_paths["C"],
    readme_path,
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in bundle_paths:
        archive.write(path, arcname=path.name)

generated = {
    "source_json": json_path,
    "source_yaml": yaml_path,
    "source_review": review_path,
    "reading_point_candidate": candidate_path,
    "rp_37_a": rp_paths["A"],
    "rp_37_b": rp_paths["B"],
    "rp_37_c": rp_paths["C"],
    "rp_37_readme": readme_path,
    "zip": zip_path,
}

generated


## Verification

This final cell verifies that every generated artifact exists and contains data.


In [ ]:
expected_dialogue_counts = {
    "A": 1,
    "B": 2,
    "C": 3,
}

for label, path in generated.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    if path.stat().st_size <= 0:
        raise ValueError(f"Empty {label}: {path}")

for stage, expected_count in expected_dialogue_counts.items():
    path = rp_paths[stage]
    loaded = yaml.safe_load(path.read_text(encoding="utf-8"))

    actual_count = len(loaded["dialogue"])
    if actual_count != expected_count:
        raise ValueError(
            f"RP_37_{stage} contains {actual_count} dialogues; "
            f"expected {expected_count}"
        )

    if loaded["identity"]["stage"] != stage:
        raise ValueError(
            f"RP_37_{stage} identity.stage does not match its filename"
        )

print("Source extraction and RP_37 YAML bundle: VERIFIED")
for label, path in generated.items():
    print(f"{label}: {path} ({path.stat().st_size} bytes)")

print()
print("Next:")
print("1. Extract RP_37_A.yaml, RP_37_B.yaml, and RP_37_C.yaml from the ZIP.")
print("2. Use each file as templates/RP_TEMPLATE.yaml.")
print("3. Run NB_TEMPLATE.ipynb for each cumulative stage.")

try:
    from google.colab import files
except ModuleNotFoundError:
    pass
else:
    files.download(str(zip_path))


## Source-Content Verification (V2)

Structural verification (above) checks that generated files exist and are
well-formed. It does not check whether the *content* of each extraction is
actually supported by its cited source page. This section does.

For each item in `EXTRACTIONS`, this cell re-opens the source PDF at the
cited page and checks that the extraction's value actually appears there
-- as a bare number for numeric extractions, or as a case-insensitive
substring for qualitative ones (e.g. "negligible"). This is a citation
check, not a semantic one: it confirms the cited page contains the
claimed value, not that the surrounding context is interpreted correctly.

Requires the source PDF (`SOURCE["source_file"]`) at
`authors-becker/Daniel Becker.pdf` relative to the repository root.

In [ ]:
import re
from pypdf import PdfReader

def _normalize_number(v):
    s = str(v)
    return s.lstrip("<>=~ ")

def v2_check_extraction(page_text: str, item: SourceExtraction) -> tuple[bool, str]:
    """Check that item.value actually appears on the cited page's text.

    Three cases, in order of how much structure the value has:
      - single bare number (e.g. 140, 1.5, "<100") -> exact digit-bounded match.
      - compound value with multiple embedded numbers (e.g. "1% in 2 minutes")
        -> every embedded number must appear somewhere on the page. This is a
        weaker check than exact-phrase matching, but exact-phrase matching is
        the wrong tool here: it would fail on any paraphrase even when every
        number checks out, which is what happened on first attempt (TS_004).
      - non-numeric qualitative value (e.g. "negligible") -> case-insensitive
        substring search.
    """
    if item.value is None:
        return True, "no value to check"
    val_str = str(item.value)
    embedded_numbers = re.findall(r'\d+(?:\.\d+)?', val_str)
    if not embedded_numbers:
        found = val_str.lower() in page_text.lower()
        return found, f"qualitative: case-insensitive search for '{val_str}' on page {item.page}"
    if len(embedded_numbers) == 1 and val_str.strip("<>=~ ") == embedded_numbers[0]:
        num = embedded_numbers[0]
        pattern = r'(?<!\d)' + re.escape(num) + r'(?!\d)'
        found = re.search(pattern, page_text) is not None
        return found, f"numeric: looked for '{num}' on page {item.page}"
    missing = [n for n in embedded_numbers if not re.search(r'(?<!\d)' + re.escape(n) + r'(?!\d)', page_text)]
    found = len(missing) == 0
    detail = f"compound: checked embedded numbers {embedded_numbers} on page {item.page}"
    if missing:
        detail += f" -- missing: {missing}"
    return found, detail

# Notebook convention (matching OUTPUT_DIRECTORY above) is that this notebook
# runs with notebooks/ as its working directory, so the source PDF lives one
# level up.
authors_dir = Path("..") / "authors-becker"
pdf_path = authors_dir / SOURCE["source_file"]
if not pdf_path.exists():
    _candidates = list(authors_dir.glob("*.pdf"))
    if len(_candidates) == 1:
        print(
            f"WARNING: SOURCE['source_file']='{SOURCE['source_file']}' "
            f"not found; falling back to the only PDF present: '{_candidates[0].name}'. "
            "SOURCE['source_file'] should be corrected to match the committed filename."
        )
        pdf_path = _candidates[0]
    else:
        raise FileNotFoundError(
            f"{pdf_path} not found and no unambiguous fallback PDF exists in {authors_dir}/."
        )

reader = PdfReader(str(pdf_path))
page_texts = {i + 1: reader.pages[i].extract_text() for i in range(len(reader.pages))}

v2_results = []
for item in EXTRACTIONS:
    text = page_texts.get(item.page, "")
    ok, detail = v2_check_extraction(text, item)
    v2_results.append({"extraction_id": item.extraction_id, "passed": ok, "detail": detail})

v2_failures = [r for r in v2_results if not r["passed"]]
display(pd.DataFrame(v2_results))

if v2_failures:
    raise ValueError(f"V2 source-content verification failed for: {[r['extraction_id'] for r in v2_failures]}")
print(f"V2 PASSED: all {len(v2_results)} extractions confirmed present on their cited pages.")

## Downstream-Consistency Verification (V3)

V2 checks that extractions match the source. This section checks that
*downstream* artifacts -- generated or hand-maintained files that repeat
source metadata (author, organization, etc.) -- agree with the canonical
`SOURCE` dict above.

This check exists specifically because it is the one that would have
caught two real errors found in `RP_37_ENGINEERING_SESSION_REPORT.md`:
the report once stated the source organization as "Idaho National
Laboratory (INL)" instead of the correct "University of Colorado" --
neither V1 (structural) nor V2 (source-content) would have caught this,
since the notebook's own `SOURCE` dict was already correct; the error was
introduced in the separately, manually maintained report. The first two
code cells below encode that as an explicit regression test.

In [ ]:
def extract_field(md_text: str, label: str) -> str | None:
    m = re.search(rf'\*\*{re.escape(label)}:\*\*\s*(.+)', md_text)
    return m.group(1).strip() if m else None

def v3_check_report(report_text: str, source: dict) -> list[str]:
    mismatches = []
    org = extract_field(report_text, "Organization")
    if org is not None and org != source["organization"]:
        mismatches.append(f"Organization: report says '{org}', SOURCE says '{source['organization']}'")
    author = extract_field(report_text, "Author")
    if author is not None and author != source["author"]:
        mismatches.append(f"Author: report says '{author}', SOURCE says '{source['author']}'")
    return mismatches

In [ ]:
# Regression test: the ORIGINAL RP_37_ENGINEERING_SESSION_REPORT.md (before
# it was corrected) stated the organization as Idaho National Laboratory.
# V3 must flag this -- if it doesn't, the check itself is broken.
_known_bad_report = '''
**Author:** Dan Becker

**Organization:** Idaho National Laboratory (INL)
'''
_regression_mismatches = v3_check_report(_known_bad_report, SOURCE)
assert len(_regression_mismatches) >= 1, (
    "V3 regression test FAILED: the known INL error was not detected. "
    "The consistency checker itself is broken."
)
print("V3 regression test passed -- checker correctly flags the known INL error:")
for m in _regression_mismatches:
    print("   ", m)

In [ ]:
# Live check: the CURRENT engineering_artifacts/RP_37_ENGINEERING_SESSION_REPORT.md
# (notebook convention: cwd = notebooks/, so this is one level up)
report_path = Path("..") / "engineering_artifacts" / "RP_37_ENGINEERING_SESSION_REPORT.md"

_report_text = report_path.read_text(encoding="utf-8")
_live_mismatches = v3_check_report(_report_text, SOURCE)

if _live_mismatches:
    print(f"V3 found {len(_live_mismatches)} mismatch(es) in the current report:")
    for m in _live_mismatches:
        print("   ", m)
    raise ValueError("V3 downstream-consistency verification failed -- see mismatches above.")
else:
    print("V3 PASSED: current report's source metadata matches SOURCE exactly.")